In [ ]:
from valdpy import ValdAuth, ForceFrameAPI
from valdpy.utils import read_credentials

%load_ext autoreload
%autoreload 2

# ForceFrame API Example

This example demonstrates how to use the VALDPY package to access ForceFrame (advanced force measurement) test data.

In [ ]:
creds = read_credentials('vald_api_cred.txt')
client_id = creds['client_id']
client_secret = creds['client_secret']
tenant_id = creds['tenant_id']
print(f"Client ID: {client_id[:10]}...")
print(f"Tenant ID: {tenant_id}")

## Step 1: Authentication

In [ ]:
auth = ValdAuth(client_id, client_secret, tenant_id=tenant_id, region='USA')

### Get OAuth Access Token

In [ ]:
token = auth.get_token()
print(f"Token obtained: {token[:20]}...")

### Optional: Get Tenant Information

In [ ]:
all_tenants = auth.get_all_tenants()
print(f"Found {len(all_tenants)} tenant(s)")

In [ ]:
tenant_info = auth.get_tenant_info()
print(f"Tenant Name: {tenant_info.get('name')}")
print(f"Tenant ID: {tenant_info.get('id')}")

## Step 2: Get Categories and Groups

In [ ]:
categories_df = auth.get_tenant_categories()
print(categories_df[['name', 'id']].to_string(index=False))

In [ ]:
groups_df = auth.get_tenant_groups()
print(f"Found {len(groups_df)} group(s)")
print(groups_df[['name', 'id']].head(10).to_string(index=False))

## Step 3: Get Profiles

In [ ]:
group_name = 'Research'
category_name = 'Team'

try:
    profiles_df = auth.get_group_profiles(group_name=group_name, category_name=category_name)
    print(f"Found {len(profiles_df)} profile(s)")
    print(profiles_df[['firstName', 'lastName', 'profileId']].head(10).to_string(index=False))
except Exception as e:
    print(f"Error: {e}")

## Step 4: Initialize ForceFrame API

In [ ]:
ff = ForceFrameAPI(tenant_id=auth.tenant_id, header=auth.header, region='USA')

### Get Test Information

In [ ]:
date = '01/01/2025'
tests_df = ff.get_tests_info(date)

if tests_df is not None:
    print(f"Found {len(tests_df)} test(s)")
    print(tests_df[['testId', 'profileId']].head())
else:
    print("No tests found")

### Get Test Results

In [ ]:
if tests_df is not None and len(tests_df) > 0:
    test_id = tests_df.iloc[0]['testId']
    results_df = ff.get_test_results(test_id)
    if results_df is not None:
        print(results_df.head())
    else:
        print("No results available")

### Visualize Results

In [ ]:
if results_df is not None:
    print("Test Results:")
    print(results_df.info())
else:
    print("No data to visualize")

In [ ]:
import matplotlib.pyplot as plt
if results_df is not None and len(results_df) > 0:
    results_df.head(10).plot(kind='bar')
    plt.title('ForceFrame Test Results')
    plt.show()
else:
    print("No data to plot")